In [14]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import math
import time

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [15]:
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = pd.read_csv('../data/processed/y_train_resampled.csv').squeeze()

X_val = np.load('../data/processed/X_val_scaled.npy')
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (2062829, 71)
X_val shape: (423051, 71)


In [16]:
class NetworkFlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

train_dataset = NetworkFlowDataset(X_train, y_train_encoded)
val_dataset = NetworkFlowDataset(X_val, y_val_encoded)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 4029
Validation batches: 827


In [17]:
class TransformerClassifier(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, num_classes, dropout=0.3):
        super(TransformerClassifier, self).__init__()
        
        self.input_projection = nn.Linear(1, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = x.unsqueeze(2)
        x = self.input_projection(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [18]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = TransformerClassifier(
    input_size=71,
    d_model=128,
    nhead=4,
    num_layers=3,
    num_classes=15,
    dropout=0.3
).to(device)

print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps
Parameters: 597,007


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [20]:
print("Training Started")
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    model.train()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{epochs} — Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Training Started
Epoch 1/10 — Loss: 0.2451, Accuracy: 91.63%
Epoch 2/10 — Loss: 0.1393, Accuracy: 94.50%


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), '../models/transformer.pth')
print("Model saved.")

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

all_preds = le.inverse_transform(all_preds)
all_labels = le.inverse_transform(all_labels)

print(classification_report(all_labels, all_preds, digits=4))